# UKCI threshold detector: demo notebook

Detects extreme and compound events across a perturbed-parameter ensemble (PPE) for a chosen UK region. 
The ensemble gives a **range of projections** rather than a single answer.

This notebook is for demonstrating how to use 'threshold_detector' to:  
 1. Detect extreme events in any cliamte timeseries
 2. Identify compound events
 3. Test whether the clustering (i.e compounds) is statistically significant (ECA)
 4. Visualise results

**You only need to change the cell below (cell 2).** Everything else runs automatically. 

## User Parametres (edit this cell only)

In [44]:
import numpy as np
import pandas as pd

# TO MOD WITH WHERE THE DATA WILL BE STORED
BASE_DIR = '/home/users/elena.dauster/Documents/python/CoinCalc/data_for_ECA'

REGION = 'Wales'   # choose one region

ENSEMBLES = ['0000','1113','1554','1649','1843','1935','2123','2242','2305','2335','2491','2868'] # do we even give this option? 

# Season 
LOWER_MONTH = 6
HIGHER_MONTH = 9

# Threshold
THRESHOLD     = 20
DIRECTION     = 'above'
N_CONSECUTIVE = 1

# Compound detection
DELT         = 4
MIN_DURATION = 2

# ECA
TAU = 1

# Plotting
CMAP           = 'Blues'
ROLLING_WINDOW = 10

# Years your data spans (for the year column)
DATA_START_YEAR = 1980

## imports (no need to touch this)

In [49]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from threshold_detector import (
    flag_extreme_events,
    detect_compound_events,
    load_region_ensemble,
    summarise_ensemble_events,
    run_eca,
    plot_ensemble_spread,
    plot_ensemble_hovmoller,
    plot_duration_and_counts,)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ImportError: cannot import name 'load_region_ensemble' from 'threshold_detector' (/home/users/elena.dauster/UKCI/threshold_detector/__init__.py)

## Step 1: Detect extreme events
`flag_extreme_events` returns a binary array: 1 where your data exceedsthe threshold, 0 elsewhere.


In [29]:
binary = flag_extreme_events(my_data, threshold=THRESHOLD, direction=DIRECTION, N=N_CONSECUTIVE)
n_extreme = binary.sum()
pct_extreme = 100 * n_extreme / len(binary)
print(f"Total timesteps:   {len(binary)}")
print(f"Extreme events:    {n_extreme}  ({pct_extreme:.1f}%)")
print(f"First 30 values:   {binary[:30]}")


Total timesteps:   14600
Extreme events:    1025  (7.0%)
First 30 values:   [0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


## Step 1b (optional): Percentile-based threshold
Use this instead of a fixed threshold if you want to adjust for warming —compute the threshold from a baseline period and apply it to the full series.   
Do i make a link to the paper? we need to explain at some point what this means and uncertainty . do we need it? 

In [30]:
# this is the percentile method, which is more robust to changing climate conditions. It uses a baseline period to calculate the threshold for extremes.
# then, it flags extremes in the full timeseries based on the value of the threshold calculated from the baseline period. 
# This is useful for detecting extremes in a changing climate, where the absolute value of extremes may change over time.

baseline_data = my_data[BASELINE_SLICE]
binary_pct, threshold_used = flag_extreme_events_percentile( timeseries=my_data, percentile=PERCENTILE, reference=baseline_data, direction=DIRECTION,)
print(f"Baseline {PERCENTILE}th percentile threshold: {threshold_used:.2f}")
print(f"Extreme events flagged: {binary_pct.sum()}")


Baseline 95th percentile threshold: 21.01
Extreme events flagged: 903


## Step 2: Detect compound events
`detect_compound_events` groups nearby extreme days into episodes.Any gap of more than `delT` days breaks a compound event into two separate ones.Events with fewer than `min_duration` extreme days are classified as single events.

the 'length' indicates how many days are involved in a compound event, regardless of whether there is an extreme event that day. 
'n_extreme_days' is how many extremes are involved in a series. 

Take this series, where 1 is an extreme and 0 is not: [0,0,0,0,1,1,0,1,0,0]    
With a tau=1, delT = 2, this series has a length = 4 and n_extreme_days = 3. 

In [31]:
events_df = detect_compound_events(binary, delT=DELT, min_duration=MIN_DURATION)
# Attach dates if available
if my_dates is not None:   
    events_df['start_date'] = my_dates[events_df['start_idx'].values]
    events_df['end_date']   = my_dates[events_df['end_idx'].values]
print(f"Compound events detected: {len(events_df)}")
print(events_df.head(10))


Compound events detected: 192
   start_idx  end_idx  length  n_extreme_days start_date   end_date
0         33       34       2               2 1980-02-03 1980-02-04
1        139      140       2               2 1980-05-19 1980-05-20
2        247      248       2               2 1980-09-04 1980-09-05
3        411      412       2               2 1981-02-15 1981-02-16
4        441      446       6               3 1981-03-17 1981-03-22
5        469      478      10               4 1981-04-14 1981-04-23
6        498      499       2               2 1981-05-13 1981-05-14
7        518      522       5               2 1981-06-02 1981-06-06
8        531      535       5               3 1981-06-15 1981-06-19
9        675      678       4               2 1981-11-06 1981-11-09


## Step 3: Test whether clustering is statistically significant (ECA)
`run_eca` tests whether extreme events cluster more than random chance predicts.- Pass the same binary series twice to test self-clustering- Pass two different series to test whether one precedes the other- p-value < 0.05 means clustering is significant

this might not always be relevant, do we include ?

In [32]:
eca_result = run_eca(binary, binary, delT=DELT, tau=TAU)
print(eca_result.summary_table())
print()
_, kt = eca_result.get_coincidences
_, _, p_prec, p_trig, _, _ = eca_result.get_poisson_values
if p_trig < 0.05:    
    print(f"✓ Clustering is SIGNIFICANT (p = {p_trig:.4f})")    
    print("  Extreme events cluster more than random chance predicts.")
else:    
    print(f"✗ Clustering is NOT significant (p = {p_trig:.4f})")    
    print("  Events are consistent with random occurrence.")


                                 Value
NH precursor                359.803082
NH trigger                  359.803082
p-value precursor             0.999905
p-value trigger               0.999905
precursor coincidence rate    0.296585
trigger coincidence rate      0.296585
p(A)                          0.070205
p(B)                          0.070205

✗ Clustering is NOT significant (p = 0.9999)
  Events are consistent with random occurrence.


### 4b — Duration and count heatmaps
These plots require an `ensemble` column in your events DataFrame.If you only have one dataset (not an ensemble), a dummy column is added.


In [40]:
import matplotlib.pyplot as plt

if 'ensemble' not in events_df.columns:    
    events_df['ensemble'] = 'single'
if 'year' not in events_df.columns and 'start_date' in events_df.columns:    
    events_df['year'] = events_df['start_date'].dt.year
elif 'year' not in events_df.columns:    
    # Approximate from index if no dates available    
    events_df['year'] = 1980 + (events_df['start_idx'] // 365).astype(int)
fig = plot_duration_and_counts( events_df, ensemble_col='ensemble', year_col='year', length_col='length', cmap=CMAP,)

#fig.savefig('../outputs/figures/duration_counts.png', dpi=150, bbox_inches='tight')
display(fig)
#print("Saved to outputs/figures/duration_counts.png")


<Figure size 1600x1000 with 4 Axes>

### 4c — Hovmöller (ensemble spread over time)
Pass a DataFrame with columns `ensemble`, `year`, and optionally`n_extreme_days` to show both event counts and extreme day counts.With a single dataset this still works — you just get one row.


In [42]:
fig_events, fig_days = plot_ensemble_hovmoller( events_df, 
    ensemble_col='ensemble', 
    year_col='year', 
    extreme_days_col='n_extreme_days' if 'n_extreme_days' in events_df.columns else None, cmap=CMAP,)

#fig_events.savefig('../outputs/figures/hovmoller_events.png', dpi=150, bbox_inches='tight')
plt.show()

if fig_days is not None:    
    #fig_days.savefig('../outputs/figures/hovmoller_days.png', dpi=150, bbox_inches='tight')
    plt.show()
print("Saved Hovmöller plots.")

Saved Hovmöller plots.


/var/tmp/ipykernel_225891/109577060.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/tmp/ipykernel_225891/109577060.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## What to do next
- To use your own data: replace `my_data` in Cell 2 with your array- To test against a second variable (e.g. atmospheric rivers vs rainfall):  
```python  
result = run_eca(ar_binary, rain_binary, delT=4, tau=0)

- To run on multiple ensemble members, loop over members and collect results into a single DataFrame before calling the plot functions
- To feed results into the full ECA analysis: open EventCoincidenceAnalysis/ECA_analysis.ipynb
